In [2]:
import os, glob, random, math
from dataclasses import dataclass
from typing import Dict, List, Tuple
from tqdm import tqdm

import numpy as np
from PIL import Image
from PIL import ImageFile

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import SegformerForSemanticSegmentation, SegformerImageProcessor

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)


DATA_ROOT = "./datasets"


CITIES = ["tunis", "manila", "copenhagen"]
YEARS_TRAIN_SEQ = [2014, 2020, 2025]  

# Your fine-tuned SegFormer checkpoint (HF folder or model id)
SEGFORMER_CKPT = "./models/segformer_osm_esri_grouped/final"


NUM_CLASSES = 7

# Tile size expected (your images should be consistent)
IMG_SIZE = 256   # 256 recommended (faster). You can use 512 if GPU allows.

# Training
BATCH_SIZE = 8
EPOCHS = 25
LR = 2e-4
SEED = 42

# Avoid decompression bomb errors
Image.MAX_IMAGE_PIXELS = None
ImageFile.LOAD_TRUNCATED_IMAGES = True

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)


c:\Users\2640870\AppData\Local\anaconda3\envs\torch_gpu\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda


In [3]:
processor = SegformerImageProcessor.from_pretrained("nvidia/segformer-b0-finetuned-ade-512-512")
segformer = SegformerForSemanticSegmentation.from_pretrained(SEGFORMER_CKPT).to(device)
segformer.eval()

# Freeze SegFormer (we only train the temporal model)
for p in segformer.parameters():
    p.requires_grad = False

@torch.no_grad()
def segformer_predict_mask(pil_img: Image.Image) -> torch.Tensor:
    """
    Returns predicted class-id mask: (H, W) torch.long on CPU
    """
    # Ensure consistent size
    pil_img = pil_img.resize((IMG_SIZE, IMG_SIZE), resample=Image.BILINEAR)

    inputs = processor(images=pil_img, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    outputs = segformer(**inputs)
    logits = outputs.logits  # (B, C, h, w) usually smaller than IMG_SIZE

    # Upsample to IMG_SIZE
    logits_up = F.interpolate(logits, size=(IMG_SIZE, IMG_SIZE), mode="bilinear", align_corners=False)
    pred = torch.argmax(logits_up, dim=1).squeeze(0).to("cpu").long()  # (H, W)

    return pred

c:\Users\2640870\AppData\Local\anaconda3\envs\torch_gpu\lib\site-packages\transformers\image_processing_base.py:417: UserWarning: The following named arguments are not valid for `SegformerImageProcessor.__init__` and were ignored: 'feature_extractor_type', 'reduce_labels'
  image_processor = cls(**image_processor_dict)


In [3]:
import os, glob, random
from typing import Dict, List, Tuple
from tqdm import tqdm

# Root folder for your data (set this somewhere above)
# DATA_ROOT = "/path/to/DATA_ROOT"
# CITIES = ["tunis", "copenhagen", "manila", ...]

IMG_EXTS = (".jpeg")  # currently only matching .jpeg files


def list_tiles_by_folder(city: str, year: int) -> Dict[str, str]:
    """
    Returns dict: tile_id -> filepath

    Expected structure:
      DATA_ROOT/<city>/<tile_id>/tiles/*<year>*.jpeg
    Example:
      tunis/out_12345/ESRI 2025.jpeg

    Note: tile_id can be any folder name, not just starting with 'out_'.
    """
    # search inside each <tile_id>/ folder for an image containing the year in its name
    # changed from "out_*" to "*" so all subfolders are considered
    pattern = os.path.join(DATA_ROOT, city, "tiles" ,"*", "*")
    files = [f for f in glob.glob(pattern) if f.lower().endswith(IMG_EXTS)]

    out: Dict[str, str] = {}
    year_str = str(year)

    for f in files:
        base = os.path.basename(f)
        if year_str not in base:
            continue

        tile_id = os.path.basename(os.path.dirname(f))  # e.g., out_12345, tile_1, etc.

        # If multiple matches exist, keep the first one; or override deterministically
        # Here: prefer "ESRI" if present, otherwise keep existing.
        if tile_id not in out:
            out[tile_id] = f
        else:
            if "esri" in base.lower() and "esri" not in os.path.basename(out[tile_id]).lower():
                out[tile_id] = f

    return out


def build_triplets() -> List[Tuple[str, str, str, str]]:
    """
    Returns list of triplets: (path_2014, path_2020, path_2025, tile_id)
    Only keeps tiles available in all 3 years.
    """
    triplets: List[Tuple[str, str, str, str]] = []

    for city in tqdm(CITIES, desc="Processing cities"):
        print(f"Processing city: {city}")
        m2014 = list_tiles_by_folder(city, 2014)
        m2020 = list_tiles_by_folder(city, 2020)
        m2025 = list_tiles_by_folder(city, 2025)

        common = sorted(set(m2014) & set(m2020) & set(m2025))
        print(city, "common tiles:", len(common))

        for tid in common:
            triplets.append((m2014[tid], m2020[tid], m2025[tid], tid))

    random.shuffle(triplets)
    return triplets


triplets = build_triplets()
print("Total triplets:", len(triplets))

split = int(0.9 * len(triplets))
train_triplets = triplets[:split]
val_triplets = triplets[split:]
print("Train:", len(train_triplets), "Val:", len(val_triplets))

Processing cities:   0%|          | 0/3 [00:00<?, ?it/s]

Processing city: tunis


Processing cities:  33%|███▎      | 1/3 [00:23<00:46, 23.14s/it]

tunis common tiles: 59143
Processing city: manila


Processing cities:  67%|██████▋   | 2/3 [00:44<00:22, 22.34s/it]

manila common tiles: 59143
Processing city: copenhagen


Processing cities: 100%|██████████| 3/3 [01:08<00:00, 22.75s/it]

copenhagen common tiles: 59143
Total triplets: 177429
Train: 159686 Val: 17743


In [4]:
from typing import Tuple
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
from tqdm.auto import tqdm
import os
import gc

Image.MAX_IMAGE_PIXELS = None

def resize_image(image: Image.Image, size: Tuple[int, int] = (256, 256)) -> Image.Image:
    return image.resize(size, resample=Image.BILINEAR)

#NOTE: segformer_predict_mask(img) should already exist and return a (H, W) long tensor.

class MaskGenDataset(Dataset):
    """
    For each index, loads 3 images and returns 3 masks:
      m2014, m2020, m2025  (H, W) class-id tensors (before one-hot)
    """
    def __init__(self, triplets):
        self.triplets = triplets

    def __len__(self):
        return len(self.triplets)

    def __getitem__(self, idx):
        p2014, p2020, p2025, tid = self.triplets[idx]

        try:
            img2014 = resize_image(Image.open(p2014).convert("RGB"))
            img2020 = resize_image(Image.open(p2020).convert("RGB"))
            img2025 = resize_image(Image.open(p2025).convert("RGB"))

            m2014 = segformer_predict_mask(img2014)  # (H, W) long
            m2020 = segformer_predict_mask(img2020)
            m2025 = segformer_predict_mask(img2025)

            # keep them on CPU, compact
            m2014 = m2014.to("cpu")
            m2020 = m2020.to("cpu")
            m2025 = m2025.to("cpu")

            return m2014, m2020, m2025

        except Exception as e:
            print(f"[SKIP] tile {tid} at idx {idx}: {e}")
            return None


In [5]:
MASK_DIR_TRAIN = "masks_train"
os.makedirs(MASK_DIR_TRAIN, exist_ok=True)

# OPTIONAL: use only a subset if you want
NUM_TILES = len(train_triplets)  # or e.g. 10000 for a subset
train_triplets_subset = train_triplets[:NUM_TILES]

mask_gen_ds = MaskGenDataset(train_triplets_subset)

# find which indices are already done (resume support)
existing = [f for f in os.listdir(MASK_DIR_TRAIN) if f.endswith(".pt")]
done_idxs = sorted(int(f.split(".")[0]) for f in existing) if existing else []

if done_idxs:
    start_idx = max(done_idxs) + 1
    print(f"Resuming mask generation from idx {start_idx} (already have {len(done_idxs)} masks)")
else:
    start_idx = 0
    print("Starting mask generation from scratch")

for idx in tqdm(range(start_idx, len(mask_gen_ds)), desc="Generating masks (TRAIN)"):
    item = mask_gen_ds[idx]
    if item is None:
        continue

    m2014, m2020, m2025 = item

    # choose compact dtype
    if NUM_CLASSES <= 255:
        dtype = torch.uint8
    else:
        dtype = torch.int16

    m2014 = m2014.to(dtype)
    m2020 = m2020.to(dtype)
    m2025 = m2025.to(dtype)

    out_path = os.path.join(MASK_DIR_TRAIN, f"{idx:06d}.pt")
    tmp_path = out_path + ".tmp"

    try:
        torch.save({"m2014": m2014, "m2020": m2020, "m2025": m2025}, tmp_path)
        os.replace(tmp_path, out_path)  # atomic rename → avoids half-written files
    except Exception as e:
        print(f"[ERROR SAVE] idx {idx}: {e}")
        if os.path.exists(tmp_path):
            os.remove(tmp_path)
        break  # stop if disk is causing problems

    # help with memory
    if (idx + 1) % 200 == 0:
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

print("Done generating TRAIN masks (or stopped on error) ✅")


Starting mask generation from scratch


Generating masks (TRAIN): 100%|██████████| 159686/159686 [4:28:14<00:00,  9.92it/s]    

Done generating TRAIN masks (or stopped on error) ✅


In [ ]:
MASK_DIR_VAL = "masks_val"
os.makedirs(MASK_DIR_VAL, exist_ok=True)

# OPTIONAL: use only a subset if you want
NUM_VAL_TILES = len(val_triplets)   
val_triplets_subset = val_triplets[:NUM_VAL_TILES]

mask_gen_val_ds = MaskGenDataset(val_triplets_subset)

# find which indices are already done (resume support)
existing = [f for f in os.listdir(MASK_DIR_VAL) if f.endswith(".pt")]
done_idxs = sorted(int(f.split(".")[0]) for f in existing) if existing else []

if done_idxs:
    start_idx = max(done_idxs) + 1
    print(f"Resuming VAL mask generation from idx {start_idx} (already have {len(done_idxs)} masks)")
else:
    start_idx = 0
    print("Starting VAL mask generation from scratch")

for idx in tqdm(range(start_idx, len(mask_gen_val_ds)), desc="Generating masks (VAL)"):
    item = mask_gen_val_ds[idx]
    if item is None:
        continue

    m2014, m2020, m2025 = item

    # choose compact dtype
    if NUM_CLASSES <= 255:
        dtype = torch.uint8
    else:
        dtype = torch.int16

    m2014 = m2014.to(dtype)
    m2020 = m2020.to(dtype)
    m2025 = m2025.to(dtype)

    out_path = os.path.join(MASK_DIR_VAL, f"{idx:06d}.pt")
    tmp_path = out_path + ".tmp"

    try:
        torch.save({"m2014": m2014, "m2020": m2020, "m2025": m2025}, tmp_path)
        os.replace(tmp_path, out_path)  # atomic rename → avoids half-written files
    except Exception as e:
        print(f"[ERROR SAVE VAL] idx {idx}: {e}")
        if os.path.exists(tmp_path):
            os.remove(tmp_path)
        break  # stop if disk is causing problems

    # help with memory
    if (idx + 1) % 200 == 0:
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

print("Done generating VAL masks (or stopped on error) ✅")


Starting VAL mask generation from scratch


Generating masks (VAL):   0%|          | 54/17743 [00:05<28:08, 10.47it/s]

In [15]:
import os
import numpy as np
import torch
from torch.utils.data import Dataset
from PIL import Image

class TrainWithPrecomputedMasks(Dataset):
    """
    Dataset that loads:
      - input image(s) from train_triplets (or from image_ds if provided)
      - precomputed masks from mask_dir/{idx:06d}.pt

    Supports common triplet formats:
      1) triplet = "path/to/image.png"
      2) triplet = ("path/to/image.png", ... )  -> uses triplet[0] as image path
      3) triplet = {"image": "..."} or {"img": "..."} or {"path": "..."} etc.
      4) triplet = (path2014, path2020, path2025) and you want to stack all years as input:
         set input_mode="stack_years"
    """

    def __init__(
        self,
        triplets,
        mask_dir="./masks_train",
        image_ds=None,
        year="m2025",
        transform=None,
        input_mode="single",   # "single" or "stack_years"
        scale_to_01=True,
    ):
        self.triplets = triplets
        self.mask_dir = mask_dir
        self.year = year
        self.transform = transform
        self.image_ds = image_ds
        self.input_mode = input_mode
        self.scale_to_01 = scale_to_01

    def __len__(self):
        return len(self.triplets)

    def _mask_path(self, idx):
        return os.path.join(self.mask_dir, f"{idx:06d}.pt")

    # ---------------------------
    # Image loading helpers
    # ---------------------------
    def _read_image_any(self, path: str) -> torch.Tensor:
        """
        Reads image and returns float32 torch tensor [C, H, W].
        Uses PIL (works for png/jpg and many tif/tiff as well).
        """
        img = Image.open(path)
        arr = np.array(img)

        # grayscale -> [H,W,1]
        if arr.ndim == 2:
            arr = arr[..., None]

        # [H,W,C] -> [C,H,W]
        x = torch.from_numpy(arr).permute(2, 0, 1).contiguous().to(torch.float32)

        if self.scale_to_01:
            mx = float(x.max().item()) if x.numel() else 0.0
            # heuristic scaling
            if mx > 1.0:
                if mx <= 255.0:
                    x = x / 255.0
                elif mx <= 65535.0:
                    x = x / 65535.0

        return x

    def _extract_path_from_triplet(self, triplet):
        """
        Supports:
          - str path
          - tuple/list with first element as path
          - dict with common keys
        """
        if isinstance(triplet, str):
            return triplet

        if isinstance(triplet, (list, tuple)):
            if len(triplet) > 0 and isinstance(triplet[0], str):
                return triplet[0]
            if len(triplet) > 0 and isinstance(triplet[0], dict):
                triplet = triplet[0]

        if isinstance(triplet, dict):
            for k in ["image", "img", "img_path", "path", "tile_path", "x_path"]:
                if k in triplet and isinstance(triplet[k], str):
                    return triplet[k]

        raise ValueError(f"Could not extract image path from triplet: {triplet}")

    def _load_image_from_triplet(self, triplet):
        """
        input_mode:
          - "single": loads ONE image from triplet (path extracted by _extract_path_from_triplet)
          - "stack_years": expects triplet = (path2014, path2020, path2025) and concatenates channels
        """
        if self.input_mode == "stack_years":
            if not (isinstance(triplet, (list, tuple)) and len(triplet) >= 3
                    and all(isinstance(p, str) for p in triplet[:3])):
                raise ValueError(
                    "input_mode='stack_years' expects triplet like (path2014, path2020, path2025). "
                    f"Got: {triplet}"
                )
            p2014, p2020, p2025 = triplet[:3]
            x2014 = self._read_image_any(p2014)
            x2020 = self._read_image_any(p2020)
            x2025 = self._read_image_any(p2025)
            x = torch.cat([x2014, x2020, x2025], dim=0)  # [C*3,H,W]
            return x

        # default: single image
        img_path = self._extract_path_from_triplet(triplet)
        return self._read_image_any(img_path)

    # ---------------------------
    # Dataset item
    # ---------------------------
    def __getitem__(self, idx):
        triplet = self.triplets[idx]

        # 1) load image
        if self.image_ds is not None:
            x = self.image_ds[idx]
            if isinstance(x, (tuple, list)):
                # common: (image, label) -> keep only image
                x = x[0]
        else:
            x = self._load_image_from_triplet(triplet)

        if self.transform is not None:
            x = self.transform(x)

        # 2) load mask
        mask_path = self._mask_path(idx)
        if not os.path.exists(mask_path):
            raise FileNotFoundError(f"Missing mask for idx={idx}: {mask_path}")

        d = torch.load(mask_path, map_location="cpu")

        if self.year in ("m2014", "m2020", "m2025"):
            y = d[self.year].long()  # for CrossEntropyLoss
            return x, y
        else:
            y2014 = d["m2014"].long()
            y2020 = d["m2020"].long()
            y2025 = d["m2025"].long()
            return x, (y2014, y2020, y2025)


In [16]:
def drop_none_collate(batch):
    batch = [b for b in batch if b is not None]
    if len(batch) == 0:
        return None
    xs, ys = zip(*batch)
    xs = torch.stack(xs, dim=0)
    ys = torch.stack(ys, dim=0)
    return xs, ys


In [18]:
from torch.utils.data import DataLoader, random_split

BATCH_SIZE = 4
VAL_RATIO = 0.2     # 20% validation
SEED = 42           # for reproducibility

dataset_size = len(triplets)  # Assuming 'triplets' is the dataset
val_size = int(dataset_size * VAL_RATIO)
train_size = dataset_size - val_size

generator = torch.Generator().manual_seed(SEED)

train_ds, val_ds = random_split(
    triplets,
    [train_size, val_size],
    generator=generator
)

print("Train samples:", len(train_ds))
print("Val samples:", len(val_ds))


NameError: name 'triplets' is not defined

In [ ]:
BATCH_SIZE = 4  

train_ds = MaskTrainDataset(MASK_DIR_TRAIN, NUM_CLASSES)
print("Number of TRAIN mask samples:", len(train_ds))

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,          
    pin_memory=True,
    collate_fn=drop_none_collate,
)


val_ds = MaskTrainDataset(MASK_DIR_VAL, NUM_CLASSES)
val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
    collate_fn=drop_none_collate,
)
